# Macks Creek Watershed — CASC2D Hydrologic Simulation

This notebook replicates the Julien, Saghafian & Ogden (1995) Macks Creek calibration example
using the **corduroy** CASC2D implementation.

**Watershed**: Macks Creek, Reynolds Creek Experimental Watershed, SW Idaho  
**Location**: ~43.0–43.25 °N, 116.85–116.5 °W  
**Event**: August 23, 1965 thunderstorm  
**Model**: Green-Ampt infiltration + 2-D diffusive-wave overland flow (no channel for this demo)

### Physics summary

| Component | Equations | Reference |
|-----------|-----------|----------|
| Green-Ampt infiltration | `f = K(1 + Hf·Md/F)` (Eq 1), mid-step quadratic (Eq 2) | Julien et al. 1995 |
| 2-D overland flow | Saint-Venant continuity (Eq 3-4), Manning q (Eq 10-11) | Julien et al. 1995 |
| ZDG boundary condition | `∂h/∂x = 0` at outlet face → free drainage | Julien et al. 1995 |

### Setup
```bash
cd /path/to/corduroy
maturin develop --features python
uv run jupyter lab notebooks/macks_creek_simulation.ipynb
```

In [1]:
import subprocess

# Build the Rust extension via uv.  Safe to re-run; no-op if already up to date.
result = subprocess.run(
    ["uv", "run", "maturin", "develop", "--features", "python"],
    capture_output=True, text=True,
    cwd="../",
)
for stream, label in [(result.stdout, "stdout"), (result.stderr, "stderr")]:
    if stream and stream.strip():
        print(f"[{label}] ...{stream[-800:]}")
if result.returncode != 0:
    raise RuntimeError(f"maturin develop failed (rc={result.returncode})")
print("Build OK")

[stdout] ...✏️ Setting installed package as editable

[stderr] ...📦 Including license file `LICENSE`
🔗 Found pyo3 bindings
🐍 Found CPython 3.13 at /home/tbindas/projects/corduroy/.venv/bin/python
Audited 5 packages in 1ms
Resolved 150 packages in 259ms
Audited 150 packages in 1ms
    Finished `dev` profile [unoptimized + debuginfo] target(s) in 0.11s
📦 Built wheel for CPython 3.13 to /tmp/.tmpyz9m7t/corduroy-0.1.0-cp313-cp313-linux_x86_64.whl
🛠 Installed corduroy-0.1.0

Build OK


In [2]:
import corduroy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
from datetime import datetime, timedelta

print(f"corduroy loaded from: {corduroy.__file__}")

corduroy loaded from: /home/tbindas/projects/corduroy/.venv/lib/python3.13/site-packages/corduroy/__init__.py


## 1  Simulation domain and parameters

USGS 3DEP elevation data is fetched via [`py3dep`](https://pypi.org/project/py3dep/),
reprojected to UTM, and resampled to the model grid resolution (152 m, matching
Julien et al. 1995).

In [3]:
# ── 3DEP DEM ───────────────────────────────────────────────────────────────
DEM_BBOX  = (-116.85, 43.0, -116.5, 43.25)   # (W, S, E, N) — Macks Creek, SW Idaho
CELL_SIZE = 152.0    # m — model resolution (Julien et al. 1995)
DEM_CRS   = "EPSG:32611"  # UTM zone 11N

# ── Simulation timestep ───────────────────────────────────────────────────
DT              = 30.0          # model timestep (s) — within CFL for 152 m cells
STEPS_PER_HOUR  = int(3600 / DT)
OUTPUT_INTERVAL = STEPS_PER_HOUR  # one snapshot per model-hour

# ── ERA5 month to scout for storms ───────────────────────────────────────
MONTH_START = "2020-10-01T00:00:00"
MONTH_END   = "2020-10-31T18:00:00"
SCOUT_STEP_HOURS  = 6
EVENT_WINDOW_HOURS = 48

# ── ERA5 bounding box (wider than watershed for surrounding ERA5 cells) ──
LAT_MIN, LAT_MAX         = 42.75, 43.50
LON_MIN_360, LON_MAX_360 = 360 - 117.25, 360 - 116.25

# ── Fetch 3DEP DEM ──────────────────────────────────────────────────────
import py3dep

print("Fetching 3DEP DEM ...")
t0 = time.time()
dem_xr = py3dep.get_dem(DEM_BBOX, resolution=30, crs=4326)
print(f"  Native: {dem_xr.shape}, CRS {dem_xr.rio.crs}  ({time.time()-t0:.1f}s)")

# Reproject to UTM at model cell size — uniform metre-spaced grid
dem_utm = dem_xr.rio.reproject(DEM_CRS, resolution=CELL_SIZE)
dem_elev = dem_utm.values.squeeze().astype(np.float64)

# Fill NoData at reprojection edges with local minimum elevation
nodata = np.isnan(dem_elev)
if nodata.any():
    dem_elev[nodata] = np.nanmin(dem_elev)
    print(f"  Filled {nodata.sum()} NoData pixels at domain edges")

NR, NC = dem_elev.shape

print(f"\nDomain  : {NR}×{NC} cells, {CELL_SIZE:.0f} m = "
      f"{NR*CELL_SIZE/1000:.1f} × {NC*CELL_SIZE/1000:.1f} km")
print(f"Elevation: {dem_elev.min():.0f}–{dem_elev.max():.0f} m "
      f"(relief {dem_elev.max()-dem_elev.min():.0f} m)")
print(f"dt      : {DT:.0f} s  ({STEPS_PER_HOUR} steps/hr)")
print(f"Scouting: {MONTH_START[:10]} → {MONTH_END[:10]}  "
      f"(6-hourly, {EVENT_WINDOW_HOURS}-h event window)")

Fetching 3DEP DEM ...
  Native: (1299, 1311), CRS EPSG:5070  (11.7s)
  Filled 21210 NoData pixels at domain edges

Domain  : 262×267 cells, 152 m = 39.8 × 40.6 km
Elevation: 687–2556 m (relief 1868 m)
dt      : 30 s  (120 steps/hr)
Scouting: 2020-10-01 → 2020-10-31  (6-hourly, 48-h event window)


## 2  ERA5 precipitation — monthly scout → peak event

Two-step approach:
1. **Scout** the full month at 6-hourly resolution (~124 timesteps, ~500 MB) to locate the
   wettest consecutive `EVENT_WINDOW_HOURS`-hour period.
2. **Fetch** that event window at hourly resolution for the simulation forcing.

ERA5 `total_precipitation` values are in **metres accumulated during the preceding hour** (m/hr).
Dividing by 3 600 converts to the m/s rate expected by `run_watershed`.

> ERA5 grid spacing is 0.25 ° (~22 km at 43 °N).  Macks Creek spans only ~8 × 8 km,
> so 4–5 ERA5 cells cover the watershed — the conservative regridder effectively
> distributes a spatially uniform rainfall onto the 53 × 53 model grid.

In [ ]:
era5_available = False
N_HOURS        = EVENT_WINDOW_HOURS   # default; overridden after ERA5 fetch
total_era5_mm  = 0.0
event_label    = "ERA5 event"
era5_lats_sub  = None
era5_lons_sub  = None
precip_era5    = None

try:
    # ── Step 1: 6-hourly scout to find peak event ───────────────────────────
    print(f"Step 1 — Scout ({MONTH_START[:10]} → {MONTH_END[:10]}, "
          f"{SCOUT_STEP_HOURS}h step) ...")
    t0 = time.time()
    ds_scout = corduroy.fetch_era5_global(
        variables       = ["total_precipitation"],
        time_start      = MONTH_START,
        time_end        = MONTH_END,
        time_step_hours = SCOUT_STEP_HOURS,
    )
    print(f"  {ds_scout}  ({time.time()-t0:.1f}s)")

    # Lat/lon arrays (reused for all fetches)
    era5_lats = np.array(ds_scout.latitudes)
    era5_lons = np.array(ds_scout.longitudes)
    lat_mask  = (era5_lats >= LAT_MIN) & (era5_lats <= LAT_MAX)
    lon_mask  = (era5_lons >= LON_MIN_360) & (era5_lons <= LON_MAX_360)
    lat_idx   = np.where(lat_mask)[0]
    lon_idx   = np.where(lon_mask)[0]
    era5_lats_sub = era5_lats[lat_mask]
    era5_lons_sub = era5_lons[lon_mask]

    # Subset to watershed bbox, replace NaN fill values with 0
    p_scout = ds_scout.get("total_precipitation")
    p_scout_sub = np.nan_to_num(
        p_scout[:, lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1].astype(np.float64),
        nan=0.0,
    )   # shape (n_6h, nlat_sub, nlon_sub)

    # Areal mean per 6-h step (m); rolling sum over EVENT_WINDOW_HOURS to find peak
    steps_in_window = EVENT_WINDOW_HOURS // SCOUT_STEP_HOURS
    areal_6h = np.mean(p_scout_sub, axis=(1, 2))          # (n_6h,)
    n_6h     = len(areal_6h)
    rolling  = np.array([
        areal_6h[i : i + steps_in_window].sum()
        for i in range(n_6h - steps_in_window + 1)
    ])
    peak_idx = int(np.argmax(rolling))
    peak_mm  = rolling[peak_idx] * 1000   # rough mm total (from sub-sampled data)

    t_month_start  = datetime.fromisoformat(MONTH_START)
    t_event_start  = t_month_start + timedelta(hours=peak_idx * SCOUT_STEP_HOURS)
    t_event_end    = t_event_start + timedelta(hours=EVENT_WINDOW_HOURS - 1)

    print(f"\n  Peak {EVENT_WINDOW_HOURS}-h window: "
          f"{t_event_start.strftime('%Y-%m-%d %H:%M')} → "
          f"{t_event_end.strftime('%Y-%m-%d %H:%M')}  "
          f"(scout total ≈ {peak_mm:.2f} mm)")

    # ── Step 2: hourly fetch for the event window ───────────────────────────
    sim_start = t_event_start.strftime("%Y-%m-%dT%H:%M:%S")
    sim_end   = t_event_end.strftime("%Y-%m-%dT%H:%M:%S")
    print(f"\nStep 2 — Hourly fetch  {sim_start} → {sim_end} ...")
    t0 = time.time()
    ds_hourly = corduroy.fetch_era5_global(
        variables       = ["total_precipitation"],
        time_start      = sim_start,
        time_end        = sim_end,
        time_step_hours = 1,
    )
    print(f"  {ds_hourly}  ({time.time()-t0:.1f}s)")

    p_hourly = ds_hourly.get("total_precipitation")
    precip_era5 = np.nan_to_num(
        p_hourly[:, lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1].astype(np.float64),
        nan=0.0,
    )   # (N_HOURS, nlat_sub, nlon_sub)  [m/hr]

    N_HOURS       = precip_era5.shape[0]
    total_era5_mm = float(np.sum(np.mean(precip_era5, axis=(1, 2))) * 1000)
    event_label   = t_event_start.strftime("%Y-%m-%d")
    era5_available = total_era5_mm > 0.1

    print(f"\n  Subset : {era5_lats_sub.shape[0]}×{era5_lons_sub.shape[0]} ERA5 cells")
    print(f"  Hours  : {N_HOURS}")
    print(f"  Areal-mean total: {total_era5_mm:.2f} mm")

except Exception as exc:
    print(f"ERA5 fetch failed: {exc}")
    print("Falling back to synthetic storm.")

In [ ]:
# ── Build hourly rainfall array (m/s), shape (N_HOURS, NR, NC) ─────────────

if era5_available:
    # ----- ERA5 path --------------------------------------------------------
    # Model grid geographic coordinates for the regridder (0-360 lon convention).
    # Target grid spans the DEM bounding box, not the wider ERA5 scout bbox.
    model_lats = np.linspace(DEM_BBOX[3] - 0.01, DEM_BBOX[1] + 0.01, NR)
    model_lons = np.linspace(DEM_BBOX[0] + 360 + 0.01, DEM_BBOX[2] + 360 - 0.01, NC)

    regridder = corduroy.ConservativeRegridder(
        era5_lats_sub, era5_lons_sub, model_lats, model_lons
    )
    print(f"Conservative regridder: "
          f"{era5_lats_sub.shape[0]}×{era5_lons_sub.shape[0]} ERA5 → {NR}×{NC} model")

    # precip_era5 is in m/hr (ERA5 hourly accumulation); convert → m/s
    rain_hourly = np.stack([
        regridder.regrid(precip_era5[t].astype(np.float32)).astype(np.float64)
        for t in range(N_HOURS)
    ])  # (N_HOURS, NR, NC)  m/hr
    rain_rate    = rain_hourly / 3600.0    # m/s
    source_label = f"ERA5 {event_label}"

else:
    # ----- Synthetic fallback: triangular 30 mm storm ----------------------
    # CASC2D paper (Julien et al. 1995): ~30 mm total, 6-hour storm, ~25 mm/hr peak.
    print(f"Using synthetic storm (ERA5 unavailable or dry).  N_HOURS = {N_HOURS}")
    rain_mm_per_hr = np.zeros(N_HOURS)
    # Triangular hyetograph: rises for hours 1-3, falls for hours 3-7
    storm_start, storm_peak, storm_end = 1, 3, 7
    peak_mm = 25.0
    for h in range(storm_start, min(storm_end, N_HOURS)):
        if h <= storm_peak:
            rain_mm_per_hr[h] = peak_mm * (h - storm_start) / (storm_peak - storm_start)
        else:
            rain_mm_per_hr[h] = peak_mm * (storm_end - h) / (storm_end - storm_peak)

    rain_rate = (
        rain_mm_per_hr[:, np.newaxis, np.newaxis] / 1000.0 / 3600.0
    ) * np.ones((N_HOURS, NR, NC))   # m/s, spatially uniform
    source_label = "Synthetic storm (30 mm triangular)"

# ── Expand hourly rate to model timesteps (dt=30s → 120 steps per hour) ───
rainfall_model = np.repeat(rain_rate, STEPS_PER_HOUR, axis=0).astype(np.float64)
# shape: (N_HOURS × STEPS_PER_HOUR, NR, NC)

print(f"\nRainfall source : {source_label}")
print(f"Array shape     : {rainfall_model.shape}")
print(f"Peak rate       : {rainfall_model.max()*3600*1000:.2f} mm/hr")
print(f"Total depth     : {rain_rate.mean(axis=(1,2)).sum()*3600*1000:.2f} mm  (domain mean)")

## 3  Soil parameters

In [ ]:
# ── Soil parameters ────────────────────────────────────────────────────────
# Crusted semi-arid rangeland with wet antecedent conditions.
# K_sat reflects biological soil crust / compacted interspace (Pierson et al. 2001,
# Reynolds Creek); M_DEF reflects near-saturated soil from prior frontal rain —
# consistent with ERA5-scale precipitation which represents synoptic events, not
# isolated convective cells.
K_SAT, H_CAP, M_DEF  = 1e-7, 0.166, 0.05    # m/s, m, dimensionless
MANN_N, RETENTION    = 0.06, 0.0025           # Manning n, retention depth (m)

k_sat_grid     = np.full((NR, NC), K_SAT)
h_cap_grid     = np.full((NR, NC), H_CAP)
m_def_grid     = np.full((NR, NC), M_DEF)
mann_n_grid    = np.full((NR, NC), MANN_N)
retention_grid = np.full((NR, NC), RETENTION)

print(f"Soil  : K={K_SAT:.2e} m/s ({K_SAT*3600*1000:.2f} mm/hr), "
      f"Hf={H_CAP} m, Md={M_DEF}, n={MANN_N}")

# Ponding depth estimate: F_p = K·Hf·Md / (i_peak − K)
i_peak = rainfall_model.max()
if i_peak > K_SAT:
    f_pond = K_SAT * H_CAP * M_DEF / (i_peak - K_SAT)
    print(f"F_ponding ≈ {f_pond*1000:.1f} mm  (ponding begins after this depth infiltrates)")
else:
    print(f"WARNING: peak rainfall rate ({i_peak*3600*1000:.2f} mm/hr) < K_sat — no Hortonian runoff possible")

# CFL check using actual DEM slopes
dx = np.diff(dem_elev, axis=1)
dy = np.diff(dem_elev, axis=0)
S_max   = max(np.abs(dx).max() / CELL_SIZE, np.abs(dy).max() / CELL_SIZE)
h_deep  = 0.10                            # assume max depth ~ 10 cm for check
V_cfl   = (1 / MANN_N) * h_deep**(2/3) * S_max**0.5
dt_max  = CELL_SIZE / (2 * V_cfl)
print(f"Max DEM slope: {S_max:.3f}")
print(f"CFL advisory: dt_max ~ {dt_max:.0f} s  (using {DT:.0f} s — "
      f"{'OK' if DT < dt_max else 'WARN: reduce DT to ~' + str(int(0.8 * dt_max)) + ' s'})")

## 4  Run the coupled model

In [ ]:
print("Running CASC2D (Green-Ampt + 2-D diffusive-wave overland flow)...")
print(f"  Steps: {len(rainfall_model)}  |  dt: {DT} s  |  "
      f"total: {len(rainfall_model)*DT/3600:.1f} h")

t_start = time.time()

result = corduroy.run_watershed(
    dem_elev        = dem_elev,
    k_sat           = k_sat_grid,
    h_cap           = h_cap_grid,
    m_def           = m_def_grid,
    mann_n          = mann_n_grid,
    retention       = retention_grid,
    rainfall        = rainfall_model,
    cell_size       = CELL_SIZE,
    dt              = DT,
    output_interval = OUTPUT_INTERVAL,
)

elapsed = time.time() - t_start
n_snap  = len(result['h_snapshots'])
h_max   = max(np.max(h) for h in result['h_snapshots'])
f_final = np.mean(result['f_snapshots'][-1]) * 1000

print(f"\nDone in {elapsed:.1f} s")
print(f"  Snapshots   : {n_snap}")
print(f"  Peak depth  : {h_max*1000:.1f} mm")
print(f"  Mean F(24h) : {f_final:.1f} mm")

## 5  Water balance and outlet hydrograph

The `run_watershed` binding currently uses overland flow only (no channel); `q_outlet` returns 0.
Outlet discharge is recovered from mass conservation:

$$Q_{\text{out,cumul}}(t) = P_{\text{cumul}}(t) - V_{\text{stored}}(t) - V_{\text{infil}}(t)$$

where all terms are in m³.

In [ ]:
cell_area = CELL_SIZE**2               # m²
times_s   = np.array(result['time'])   # seconds
times_hr  = times_s / 3600.0           # hours

# Stored overland volume at each snapshot
V_stored = np.array([np.sum(h) * cell_area for h in result['h_snapshots']])  # m³

# Cumulative infiltrated volume at each snapshot
V_infil  = np.array([np.sum(f) * cell_area for f in result['f_snapshots']])  # m³

# Cumulative rainfall volume at each snapshot
# Snapshot i ends after (i+1)*OUTPUT_INTERVAL model steps
P_cumul = np.array([
    np.sum(rainfall_model[: (i + 1) * OUTPUT_INTERVAL]) * DT * cell_area
    for i in range(n_snap)
])  # m³

Q_out_cumul = np.maximum(0.0, P_cumul - V_stored - V_infil)  # m³
Q_out_rate  = np.gradient(Q_out_cumul, times_s)               # m³/s

# Domain-average depths in mm
domain_area   = NR * NC * cell_area
rain_cumul_mm = P_cumul / domain_area * 1000
infil_mm      = V_infil / domain_area * 1000
stored_mm     = V_stored / domain_area * 1000
runoff_mm     = Q_out_cumul / domain_area * 1000

print("Water balance at t = 24 h:")
print(f"  Rainfall input   : {rain_cumul_mm[-1]:.2f} mm")
print(f"  Cumul. infiltr.  : {infil_mm[-1]:.2f} mm  ({100*infil_mm[-1]/max(rain_cumul_mm[-1],1e-9):.0f} %)")
print(f"  Overland storage : {stored_mm[-1]:.2f} mm")
print(f"  Runoff (outlet)  : {runoff_mm[-1]:.2f} mm  ({100*runoff_mm[-1]/max(rain_cumul_mm[-1],1e-9):.0f} %)")
print(f"  Peak discharge   : {Q_out_rate.max():.1f} m³/s  at t = {times_hr[np.argmax(Q_out_rate)]:.1f} h")

## 6  Visualisation

In [ ]:
# ── Figure 1: hyetograph + hydrograph ──────────────────────────────────────
fig = plt.figure(figsize=(12, 7))
gs  = gridspec.GridSpec(3, 1, hspace=0.05, figure=fig)

# — Hyetograph (top)
ax_rain = fig.add_subplot(gs[0])
rain_mm_hr = rain_rate.mean(axis=(1, 2)) * 3600 * 1000   # domain-mean, mm/hr
ax_rain.bar(np.arange(N_HOURS) + 0.5, rain_mm_hr, width=0.85,
            color='steelblue', alpha=0.75, label=source_label)
ax_rain.invert_yaxis()   # conventional hyetograph orientation
ax_rain.set_ylabel("Rainfall (mm/hr)", fontsize=10)
ax_rain.set_xlim([0, N_HOURS])
ax_rain.tick_params(labelbottom=False)
ax_rain.legend(loc='lower right', fontsize=9)
ax_rain.set_title(f"Macks Creek — August 23, 1965  "
                  f"[total: {rain_mm_hr.sum():.1f} mm, "
                  f"runoff coeff: {runoff_mm[-1]/max(rain_cumul_mm[-1],1e-9):.2f}]",
                  fontsize=11)

# — Hydrograph (bottom two rows)
ax_q = fig.add_subplot(gs[1:])
ax_q.plot(times_hr, Q_out_rate, color='firebrick', linewidth=2,
          label='Outlet Q (mass balance)')
ax_q.fill_between(times_hr, 0, Q_out_rate, alpha=0.20, color='firebrick')
ax_q.axvline(times_hr[np.argmax(Q_out_rate)], color='firebrick', linestyle='--',
             alpha=0.5, linewidth=1, label=f'Peak {Q_out_rate.max():.1f} m³/s')
ax_q.set_xlabel("Time (hours)", fontsize=10)
ax_q.set_ylabel("Discharge (m³/s)", fontsize=10)
ax_q.set_xlim([0, N_HOURS])
ax_q.set_ylim(bottom=0)
ax_q.grid(True, alpha=0.3)
ax_q.legend(fontsize=9)

plt.savefig("macks_creek_hydrograph.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: macks_creek_hydrograph.png")

In [ ]:
# ── Figure 2: depth snapshots ───────────────────────────────────────────────
snap_hrs  = [3, 6, 12, 23]
snap_idxs = [min(h, n_snap - 1) for h in snap_hrs]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Overland Water Depth — Macks Creek", fontsize=13)

x_km = np.linspace(0, NC * CELL_SIZE / 1000, NC)
y_km = np.linspace(NR * CELL_SIZE / 1000, 0, NR)   # row 0 = upstream (top)

for ax, h_idx, hr in zip(axes.flatten(), snap_idxs, snap_hrs):
    h_map = result['h_snapshots'][h_idx]
    h_max_snap = h_map.max()
    vmax  = max(h_max_snap, 1e-4)

    im = ax.pcolormesh(x_km, y_km, h_map, cmap='Blues',
                       vmin=0, vmax=vmax, shading='auto')
    # DEM contours
    ax.contour(x_km, y_km, dem_elev, levels=10,
               colors='k', alpha=0.35, linewidths=0.6)
    # Outlet location
    ax.plot(NC // 2 * CELL_SIZE / 1000, 0, 'rv', markersize=8, label='outlet')

    plt.colorbar(im, ax=ax, label='h (m)')
    ax.set_title(f"t = {hr:02d} h  |  max h = {h_max_snap*1000:.1f} mm", fontsize=10)
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")

plt.tight_layout()
plt.savefig("macks_creek_depth_maps.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: macks_creek_depth_maps.png")

In [ ]:
# ── Figure 3: water balance + final infiltration ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Macks Creek — Water Balance & Infiltration", fontsize=12)

# — Left: cumulative depth time series
ax = axes[0]
ax.plot(times_hr, rain_cumul_mm, 'b-',  linewidth=2, label='Cumul. rainfall')
ax.plot(times_hr, infil_mm,      'g--', linewidth=2, label='Cumul. infiltration')
ax.plot(times_hr, runoff_mm,     'r-',  linewidth=2, label='Cumul. runoff')
ax.plot(times_hr, stored_mm,     'k:',  linewidth=1.5, label='Stored overland')
ax.set_xlabel("Time (hours)")
ax.set_ylabel("Depth (mm)")
ax.set_title("Domain-average water balance")
ax.legend()
ax.grid(True, alpha=0.3)

# — Right: final cumulative infiltration depth map
ax2 = axes[1]
f_final_mm = result['f_snapshots'][-1] * 1000
im = ax2.pcolormesh(x_km, y_km, f_final_mm, cmap='YlOrBr', shading='auto')
ax2.contour(x_km, y_km, dem_elev, levels=8, colors='k', alpha=0.3, linewidths=0.5)
ax2.plot(NC // 2 * CELL_SIZE / 1000, 0, 'rv', markersize=8)
plt.colorbar(im, ax=ax2, label='F (mm)')
ax2.set_title(f"Cumulative infiltration at t = 24 h (mean {f_final_mm.mean():.1f} mm)")
ax2.set_xlabel("x (km)")
ax2.set_ylabel("y (km)")

plt.tight_layout()
plt.savefig("macks_creek_water_balance.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: macks_creek_water_balance.png")

## 7  Summary

| Component | Implementation | Notes |
|-----------|---------------|-------|
| Precipitation | ERA5 (ARCO Zarr v3, GCS) | `corduroy.fetch_era5_global` — parallel tokio fetch |
| Regridding | Conservative area-weighted | `corduroy.ConservativeRegridder` — Rust |
| Infiltration | Green-Ampt Eq 1-2 | Per-cell, mid-timestep quadratic |
| Overland flow | 2-D diffusive wave Eq 3-11 | Manning, upwind depth, ZDG outlet BC |
| Hydrograph | Mass balance Q = P − dV/dt | Channel routing not yet wired into Python binding |

### Next steps
- Add channel routing (`ChannelReach` / `ChannelNetwork`) to the `run_watershed` Python binding  
- Replace synthetic DEM with actual 10-m NED tiles for Reynolds Creek  
- Calibrate K, Hf, Md, and Manning n against observed 1965 storm hydrograph  
- Extend to multi-day/multi-event runs using ERA5 hourly precipitation